# Hyperspectral training (Spectral MAE / ViT)

Default `MODEL_FAMILY = "spectral_mae"` matches the GHG paper band-wise masking and MAE loss on masked bands. Use `"vit_reconstruction"` for the spatial-patch ViT baseline.

Set `DATASET_ROOT` to a directory built with `download_dataset.run_full_dataset_build`. For a dry run without EMIT chips, synthetic data is used when `dataset_index.csv` is missing.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# encoder_refactor on path
_ROOT = Path("/workspace/Hyperspectral_Coolstuff/encoder_refactor").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader

from utilities import (
    EmitHypercubeIndexDataset,
    band_indices_to_mask,
    band_masked_l1,
    collate_hypercube_batch,
    linear_warmup_cosine_scheduler,
    load_band_stats_npz,
    load_dataset_index_csv,
    load_wavelengths_nm_from_instrument,
    masked_band_mse,
    masked_l1_full_cube,
    num_bands_from_instrument,
    save_checkpoint,
    set_seed,
    train_val_split_by_granule,
)
from model_definition import ModelConfig, build_model, model_config_to_dict
from presets import build_spectral_mae_preset, build_vit_preset
from spectral_mae import build_spectral_mae_model, spectral_mae_config_to_dict

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

In [ ]:
# --- User configuration ---------------------------------------------------

DATASET_ROOT = Path("/workspace/Hyperspectral_Coolstuff/emit_dataset")  # change me
INDEX_CSV = DATASET_ROOT / "dataset_index.csv"
BAND_STATS_NPZ = DATASET_ROOT / "emit_band_stats.npz"  # optional; omit normalization if missing

USE_SYNTHETIC_DATA = not INDEX_CSV.is_file()

# "spectral_mae" = GHG paper band masking + MAE loss; "vit_reconstruction" = spatial-patch ViT + MSE
MODEL_FAMILY = "spectral_mae"

SPATIAL_SIZE = (256, 256)
IN_CHANNELS = None  # filled from instrument.json when using real data
BATCH_SIZE = 2
NUM_EPOCHS = 3
LR = 3e-4
WEIGHT_DECAY = 0.05
SEED = 42
VAL_FRACTION = 0.15
CHECKPOINT_PATH = _ROOT / "checkpoints" / (
    "spectral_mae_latest.pt" if MODEL_FAMILY == "spectral_mae" else "vit_hs_latest.pt"
)

set_seed(SEED)

In [ ]:
from torch.utils.data import Dataset


class SyntheticCubeDataset(Dataset):
    """Random NCHW cubes for smoke-training when no dataset_index.csv exists."""

    def __init__(self, n: int, channels: int, spatial: tuple[int, int]):
        self.n = n
        self.channels = channels
        self.spatial = spatial

    def __len__(self):
        return self.n

    def __getitem__(self, idx: int):
        h, w = self.spatial
        x = torch.randn(self.channels, h, w)
        valid = torch.ones_like(x)
        meta = {"id": f"synth_{idx}", "label": "synthetic"}
        return {"cube": x, "valid": valid, "meta": meta}


def collate_synth(samples):
    cubes = torch.stack([s["cube"] for s in samples])
    valids = torch.stack([s["valid"] for s in samples])
    return {"cube": cubes, "valid": valids, "meta": [s["meta"] for s in samples]}


if USE_SYNTHETIC_DATA:
    print("Using synthetic cubes (no dataset_index.csv at", INDEX_CSV, ")")
    IN_CHANNELS = 32
    train_ds = SyntheticCubeDataset(64, IN_CHANNELS, SPATIAL_SIZE)
    val_ds = SyntheticCubeDataset(16, IN_CHANNELS, SPATIAL_SIZE)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_synth, num_workers=0
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_synth, num_workers=0)
else:
    IN_CHANNELS = num_bands_from_instrument(DATASET_ROOT)
    df = load_dataset_index_csv(INDEX_CSV)
    df_train, df_val = train_val_split_by_granule(df, VAL_FRACTION, SEED)
    band_mean = band_std = None
    if BAND_STATS_NPZ.is_file():
        band_mean, band_std = load_band_stats_npz(BAND_STATS_NPZ)
    train_ds = EmitHypercubeIndexDataset(
        df_train, DATASET_ROOT, band_mean=band_mean, band_std=band_std
    )
    val_ds = EmitHypercubeIndexDataset(df_val, DATASET_ROOT, band_mean=band_mean, band_std=band_std)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_hypercube_batch,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_hypercube_batch,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

print("train batches", len(train_loader), "val batches", len(val_loader), "C", IN_CHANNELS)

In [ ]:
# Wavelengths for Spectral MAE positional encoding (paper Eq. 7–9)
if USE_SYNTHETIC_DATA:
    WAVELENGTHS_NM = torch.linspace(400.0, 2500.0, IN_CHANNELS)
else:
    WAVELENGTHS_NM = torch.tensor(
        load_wavelengths_nm_from_instrument(DATASET_ROOT), dtype=torch.float32
    )

assert SPATIAL_SIZE[0] % 16 == 0 and SPATIAL_SIZE[1] % 16 == 0, "spatial size must divide patch_size 16"

model_cfg = None
mae_cfg = None

if MODEL_FAMILY == "spectral_mae":
    mae_cfg = build_spectral_mae_preset(
        "ghg_paper_spectral_mae",
        spatial_size=SPATIAL_SIZE,
        in_channels=IN_CHANNELS,
        patch_size=16,
    )
    model = build_spectral_mae_model(mae_cfg).to(DEVICE)
    print(mae_cfg)
else:
    model_cfg = build_vit_preset(
        "vit_patch16_emit256",
        spatial_size=SPATIAL_SIZE,
        in_channels=IN_CHANNELS,
        patch_size=16,
    )
    model = build_model(model_cfg).to(DEVICE)
    print(model_cfg)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = max(1, NUM_EPOCHS * len(train_loader))
warmup_steps = min(500, total_steps // 10)
scheduler = linear_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps)

In [ ]:
WL = WAVELENGTHS_NM.to(DEVICE)


def train_one_epoch():
    model.train()
    losses = []
    for batch in train_loader:
        x = batch["cube"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        if MODEL_FAMILY == "spectral_mae":
            out = model(x, wavelengths_nm=WL)
            pred = out.cube_reconstruction
            bm = band_indices_to_mask(IN_CHANNELS, out.mask_band_indices, device=DEVICE)
            loss = band_masked_l1(pred, x, valid, bm)
        else:
            out = model(x)
            pred = out.cube_reconstruction
            if pred is None:
                raise RuntimeError("ViT model has no reconstruction head")
            loss = masked_band_mse(pred, x, valid)
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for batch in val_loader:
        x = batch["cube"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)
        if MODEL_FAMILY == "spectral_mae":
            out = model(x, wavelengths_nm=WL)
            pred = out.cube_reconstruction
            bm = band_indices_to_mask(IN_CHANNELS, out.mask_band_indices, device=DEVICE)
            losses.append(band_masked_l1(pred, x, valid, bm).item())
        else:
            out = model(x)
            pred = out.cube_reconstruction
            losses.append(masked_band_mse(pred, x, valid).item())
    return float(np.mean(losses))


best_val = float("inf")
for epoch in range(NUM_EPOCHS):
    tr = train_one_epoch()
    va = evaluate()
    print(f"epoch {epoch+1}/{NUM_EPOCHS}  train {tr:.6f}  val {va:.6f}")
    if va < best_val:
        best_val = va
        CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        if MODEL_FAMILY == "spectral_mae":
            cfg_dict = spectral_mae_config_to_dict(mae_cfg)
            mf = "spectral_mae"
        else:
            cfg_dict = model_config_to_dict(model_cfg)
            mf = "vit_reconstruction"
        save_checkpoint(
            CHECKPOINT_PATH,
            model_state_dict=model.state_dict(),
            model_config_dict=cfg_dict,
            optimizer_state_dict=optimizer.state_dict(),
            epoch=epoch,
            extra={"best_val": best_val, "model_family": mf},
        )
        print("  saved", CHECKPOINT_PATH)

print("done. best val", best_val)